In [ ]:
from pathlib import Path

import polars as pl
from loguru import logger

from scrapetube.config import (
    QUERY_STRINGS,
    VIDEO_BASE_URL,
    KEYWORD_VIDEOS_META_PARQUET,
    CHANNEL_VIDEOS_META_PARQUET,
    SUBTITLES_PARQUET,
    get_today_string
)
from scrapetube.scrapetube_cc import collect_and_save_video_metadata_concurrent
from scrapetube.scrapetube_cc_subtitle import get_video_transcripts_concurrent
from scrapetube.scrapetube_cc_text import convert_subtitles_to_text, create_items_from_dataframe
from scrapetube.scrapetube_cc_license import process_videos_concurrent
from scrapetube.logger import setup_logging

pl.Config.set_fmt_str_lengths(100)

today_string = get_today_string()
setup_logging(write_to_file=False)

In [ ]:
def load_latest_data(prefix:str, data_dir:Path):
    """Load the most recently modified parquet file with the given prefix from data directory."""
    latest_file = max(data_dir.glob(f"{prefix}*.parquet"), key=lambda f: f.stat().st_mtime)
    logger.info(f"Loaded {latest_file}")
    df = pl.read_parquet(latest_file)
    return df

In [ ]:
QUERY_STRINGS[:5]

In [ ]:
KEYWORD_VIDEOS_META_PARQUET

In [ ]:
data_dir = Path("../data")

In [ ]:
params = {
    "queries": QUERY_STRINGS[:3],
    "limit": 10,
    "sleep": (3, 20),
    "sp_filter": "relevance",
    "results_type": "video",
    "proxies": None,
    "video_base_url": VIDEO_BASE_URL,
    "file_path_parquet": KEYWORD_VIDEOS_META_PARQUET,
}
results = collect_and_save_video_metadata_concurrent(**params)

In [ ]:
video_meta_df = load_latest_data("meta_data", data_dir)
unique_channel = [f'"{c}"' for c in video_meta_df["channel"].unique()]
unique_channel

In [ ]:
params = {
    "queries": unique_channel,
    "limit": 5,
    "sleep": (3, 20),
    "sp_filter": "relevance",
    "results_type": "video",
    "proxies": None,
    "video_base_url": VIDEO_BASE_URL,
    "file_path_parquet": CHANNEL_VIDEOS_META_PARQUET,
}
results = collect_and_save_video_metadata_concurrent(**params)

In [ ]:
channel_video_meta_df = load_latest_data("channel_video_meta", data_dir)
channel_video_meta_df

In [ ]:
_ = get_video_transcripts_concurrent(
    video_ids=channel_video_meta_df["video_id"].to_list(),
    file_path_parquet=SUBTITLES_PARQUET,
    sleep=(5, 25),
)

In [ ]:
subtitle_df = load_latest_data("subtitle", data_dir)
subtitle_df

In [ ]:
channel_df_with_sub = channel_video_meta_df.join(
    subtitle_df, on="video_id", how="inner"
).with_columns(pl.col("subtitles").str.json_decode().alias("subtitles_json"))
channel_df_with_sub = convert_subtitles_to_text(channel_df_with_sub)

In [ ]:
check_license_result = process_videos_concurrent(
    video_urls=channel_df_with_sub["video_url"].to_list(),
    sleep=(5, 25),
)

In [ ]:
channel_df_with_sub = channel_df_with_sub.with_columns(
    pl.Series([item["license"] for item in check_license_result]).alias("license")
).filter(
    pl.col("license")
    .str.strip_chars()
    .str.to_lowercase()
    .str.contains("creative commons")
)

In [ ]:
create_items_from_dataframe(channel_df_with_sub)